# 21.8 数据存储格式:行式 vs 列式 / Data Storage Formats: Row vs Columnar (CSV, Parquet, Arrow)

**中文**:前几节 Polars/DuckDB 的惊人速度,有一半功劳其实属于**数据的存储格式**。同样的数据,存成 **CSV** 还是 **Parquet**,读取速度可能差**几十倍**、体积差**几倍**。这不是玄学,而是**行式存储 vs 列式存储**的根本差异。这是数据工程最基础也最高频的面试考点之一——*"为什么大数据都用 Parquet 而不是 CSV?"* 本节用**真实基准**量化答案,拆解 Parquet 的内部结构(行组、列块、统计信息),并讲清 **Apache Arrow** 为什么是连接 pandas/Polars/DuckDB/Spark 的"内存通用语"。搞懂存储格式,你才真正理解现代数据栈为什么快。
**English**: Half the credit for Polars/DuckDB's stunning speed in prior sections actually belongs to the **data storage format**. The same data stored as **CSV** vs **Parquet** can differ by **tens of times** in read speed and **several times** in size. This isn't magic but the fundamental difference between **row-based and columnar storage**. It's one of data engineering's most basic and frequent interview topics — *"why does big data use Parquet instead of CSV?"* This section quantifies the answer with **real benchmarks**, dissects Parquet's internals (row groups, column chunks, statistics), and explains why **Apache Arrow** is the "in-memory lingua franca" connecting pandas/Polars/DuckDB/Spark. Understand storage formats and you truly understand why the modern data stack is fast.

---

**中文**:**行式 vs 列式(一张图讲清)**。假设一张表有 `id, cat, val, flag` 四列:
**English**: **Row vs columnar (one picture)**. Suppose a table has four columns `id, cat, val, flag`:
- **中文**:**行式存储(CSV、Avro、传统数据库)**:一行的所有字段连续存放:`[行1的id,cat,val,flag][行2的id,cat,val,flag]…`。适合"一次读写整行"(OLTP 事务),但分析时"只求 val 列的平均值"却**被迫读入所有列**。
  **Row-based (CSV, Avro, traditional DBs)**: all fields of a row are stored contiguously: `[row1 id,cat,val,flag][row2 id,cat,val,flag]…`. Good for "read/write a whole row at once" (OLTP transactions), but for analytics "just average the val column" you are **forced to read all columns**.
- **中文**:**列式存储(Parquet、ORC、Arrow)**:同一列的数据连续存放:`[所有id][所有cat][所有val][所有flag]`。分析时**只读需要的列**(列裁剪),而且同列数据类型相同、取值相似,**压缩率极高**(字典编码、RLE)。
  **Columnar (Parquet, ORC, Arrow)**: all values of a column are stored contiguously: `[all ids][all cats][all vals][all flags]`. For analytics you **read only the needed columns** (column pruning), and same-column data (same type, similar values) **compresses extremely well** (dictionary encoding, RLE).

**中文**:**列式为什么对分析这么关键**:①**列裁剪**——只查 2 列就只读 2 列的字节,不碰其他列;②**高压缩**——`cat` 列全是 "apple/banana"这种低基数值,字典编码后极小;③**谓词下推**——Parquet 每个"行组"存了各列的 min/max 统计,`WHERE val>1000` 时能**直接跳过 max<1000 的整个行组**,连读都不读。这三点叠加,就是 Parquet 比 CSV 快几十倍的原因。
**English**: **Why columnar is so key for analytics**: ① **column pruning** — querying 2 columns reads only those 2 columns' bytes, never touching others; ② **high compression** — a `cat` column of low-cardinality values like "apple/banana" becomes tiny after dictionary encoding; ③ **predicate pushdown** — Parquet stores per-column min/max statistics for each "row group," so `WHERE val>1000` can **skip entire row groups where max<1000** without even reading them. These three combined are why Parquet is tens of times faster than CSV.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 数据工程必考）**
> **中文**:**行式(CSV/Avro/OLTP库)**:整行连续存, 适合逐行读写/事务/流式追加; 分析时被迫读全部列。**列式(Parquet/ORC/Arrow)**:整列连续存, 适合分析——①列裁剪(只读需要列)②高压缩(同列同类型, 字典/RLE 编码)③谓词下推(行组 min/max 统计→跳过无关行组)。**Parquet**=事实标准的**磁盘**列式格式(带 schema、压缩、行组统计、支持嵌套)。**Arrow**=**内存**列式标准(零拷贝在 pandas/Polars/DuckDB/Spark 间传数据, 是它们的"通用语")；Parquet↔Arrow 高效互转。**Avro**=行式、强 schema 演进、适合 Kafka/流式(逐行写)。**ORC**=列式, Hive 生态。**CSV**=人类可读、通用交换, 但无类型、无压缩、慢——别用于大数据存储。**选型**:分析/数仓/数据湖→**Parquet**; 内存中跨引擎传输→**Arrow**; 流式/消息/需 schema 演进→**Avro**; 小数据交换/给人看→CSV。**分区(partitioning)**:按列(如 date=2024-01-01/)把 Parquet 分目录存→查询时分区裁剪只读相关目录。面试金句:*"列式(Parquet)比行式(CSV)快在分析场景:只读需要的列(列裁剪)、同列高压缩、行组统计支持谓词下推跳过无关数据; Parquet 是磁盘列式标准、Arrow 是内存列式标准(零拷贝跨引擎); 流式/schema演进用行式的 Avro; 大数据别用 CSV。"*
> **English**: **Row-based (CSV/Avro/OLTP DBs)**: whole rows stored contiguously, good for row-wise read/write / transactions / streaming appends; analytics is forced to read all columns. **Columnar (Parquet/ORC/Arrow)**: whole columns contiguous, good for analytics — ① column pruning (read only needed columns) ② high compression (same-column same-type, dictionary/RLE encoding) ③ predicate pushdown (row-group min/max statistics → skip irrelevant row groups). **Parquet** = the de-facto **on-disk** columnar format (with schema, compression, row-group stats, nested support). **Arrow** = the **in-memory** columnar standard (zero-copy data passing among pandas/Polars/DuckDB/Spark, their "lingua franca"); Parquet↔Arrow convert efficiently. **Avro** = row-based, strong schema evolution, good for Kafka/streaming (row-wise writes). **ORC** = columnar, Hive ecosystem. **CSV** = human-readable, universal interchange, but untyped, uncompressed, slow — don't use for big-data storage. **Tool choice**: analytics/warehouse/data-lake → **Parquet**; in-memory cross-engine transfer → **Arrow**; streaming/messaging/schema evolution → **Avro**; small interchange/human-readable → CSV. **Partitioning**: store Parquet in directories by a column (e.g. date=2024-01-01/) → partition pruning reads only relevant directories. Interview line: *"Columnar (Parquet) beats row-based (CSV) for analytics: read only needed columns (column pruning), high same-column compression, and row-group statistics enabling predicate pushdown to skip irrelevant data; Parquet is the on-disk columnar standard, Arrow the in-memory one (zero-copy cross-engine); use row-based Avro for streaming/schema evolution; don't use CSV for big data."*


In [ ]:

# ============================================================
# 真实基准:CSV vs Parquet 的体积与读取速度 / REAL benchmark: CSV vs Parquet size & read speed
# ============================================================
import numpy as np, pandas as pd, pyarrow.parquet as pq, time, os
np.random.seed(0)
N=3_000_000
df=pd.DataFrame({"id":np.arange(N),
                 "cat":np.random.choice(["apple","banana","cherry","date"],N),  # 低基数→压缩好 / low-cardinality
                 "val":np.random.rand(N)*100,
                 "flag":np.random.randint(0,2,N).astype(bool)})
csv="/tmp/fmt.csv"; pqt="/tmp/fmt.parquet"
df.to_csv(csv,index=False)
df.to_parquet(pqt,compression="snappy")                    # Parquet + snappy 压缩 / compressed
mb=lambda p: os.path.getsize(p)/1e6
print(f"体积 / size:  CSV {mb(csv):6.1f} MB   Parquet {mb(pqt):6.1f} MB   → Parquet 小 {mb(csv)/mb(pqt):.1f}x")
def bench(f,rep=3): return min(((lambda t0=time.time():(f(),time.time()-t0)[1])()) for _ in range(rep))
# 读全表 / read all columns
t_csv=bench(lambda: pd.read_csv(csv)); t_pq=bench(lambda: pd.read_parquet(pqt))
# 只读 1 列(列式的主场)/ read only 1 column (columnar's strength)
t_csv1=bench(lambda: pd.read_csv(csv,usecols=["val"]))
t_pq1 =bench(lambda: pd.read_parquet(pqt,columns=["val"]))
print(f"读全表 read-all : CSV {t_csv*1000:6.0f} ms   Parquet {t_pq*1000:6.0f} ms   → 快 {t_csv/t_pq:4.1f}x")
print(f"只读 1 列 read-1col: CSV {t_csv1*1000:6.0f} ms   Parquet {t_pq1*1000:6.0f} ms   → 快 {t_csv1/t_pq1:4.1f}x (列式只读该列的字节!)")
# Parquet 的行组统计(谓词下推的基础)/ Parquet row-group statistics (basis of predicate pushdown)
meta=pq.ParquetFile(pqt).metadata
rg=meta.row_group(0).column(2)                             # val 列在第0个行组的统计 / stats for val col
print(f"\nParquet 内部:{meta.num_row_groups} 个行组(row group), 每个存各列 min/max。")
print(f"  例:第0行组 val 列 min={rg.statistics.min:.2f}, max={rg.statistics.max:.2f}")
print("  → WHERE val>99 时, max<99 的整个行组直接跳过不读(predicate pushdown), CSV 做不到")


In [ ]:

# ============================================================
# Apache Arrow:内存列式标准 + 零拷贝跨引擎 / Arrow: in-memory columnar standard + zero-copy interop
# 中文:Arrow 是"内存里的 Parquet"——pandas/Polars/DuckDB/Spark 都用它做内存格式, 于是彼此传数据可零拷贝。
# English: Arrow is "Parquet in memory" — pandas/Polars/DuckDB share it as their memory format, so they pass data zero-copy.
# ============================================================
import pyarrow as pa, polars as pl, duckdb
# pandas → Arrow(常零拷贝)→ Polars / pandas → Arrow (often zero-copy) → Polars
arrow_tbl=pa.Table.from_pandas(df)                         # DataFrame 转 Arrow 表 / to Arrow table
pl_from_arrow=pl.from_arrow(arrow_tbl)                     # Arrow → Polars(零拷贝共享底层内存)/ zero-copy
print("Arrow 表 schema / Arrow table schema:", [f"{f.name}:{f.type}" for f in arrow_tbl.schema])
print("同一份 Arrow 内存被 Polars 直接接手(零拷贝)/ Polars adopts the same Arrow memory (zero-copy):", pl_from_arrow.shape)
# DuckDB 直接查 Arrow 表(不复制数据)/ DuckDB queries the Arrow table directly (no copy)
res=duckdb.sql("SELECT cat, count(*) n, avg(val) FROM arrow_tbl GROUP BY cat ORDER BY cat").pl()
print("\nDuckDB 直接对 Arrow 表跑 SQL(pandas/Polars/DuckDB 共享 Arrow→无缝互操作):")
print(res)
print("→ Arrow 是现代数据栈的'内存通用语':一份数据在多引擎间流转而无需反复序列化/复制")


In [ ]:

# ============================================================
# 可视化:体积、读取速度、行vs列布局 / size, read speed, row vs columnar layout
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
# ① 体积 / size
ax[0].bar(["CSV","Parquet"],[mb(csv),mb(pqt)],color=["#C44E52","#55A868"])
for i,v in enumerate([mb(csv),mb(pqt)]): ax[0].text(i,v+1,f"{v:.0f}MB",ha="center",fontsize=11,weight="bold")
ax[0].set_title(f"体积:Parquet 小 {mb(csv)/mb(pqt):.1f}x"); ax[0].set_ylabel("MB")
# ② 读取速度 / read speed
x=np.arange(2); w=0.35
ax[1].bar(x-w/2,[t_csv*1000,t_csv1*1000],w,label="CSV",color="#C44E52")
ax[1].bar(x+w/2,[t_pq*1000,t_pq1*1000],w,label="Parquet",color="#55A868")
ax[1].set_xticks(x); ax[1].set_xticklabels(["读全表","只读1列"]); ax[1].set_ylabel("ms(越低越好)")
ax[1].set_title("读取速度:只读1列时列式碾压"); ax[1].legend()
# ③ 行vs列布局 / row vs columnar layout
ax[2].axis("off"); ax[2].set_title("行式 vs 列式 磁盘布局",fontsize=11,weight="bold")
ax[2].text(0.5,0.9,"行式(CSV):整行连续",ha="center",fontsize=9,color="#C44E52",transform=ax[2].transAxes)
cols=["#DD8452","#4C72B0","#55A868","#C44E52"]
for r in range(2):
    for c in range(4):
        ax[2].add_patch(plt.Rectangle((0.05+c*0.11,0.72-r*0.09),0.1,0.08,fc=cols[c],alpha=0.7,transform=ax[2].transAxes))
ax[2].text(0.5,0.5,"列式(Parquet):整列连续",ha="center",fontsize=9,color="#55A868",transform=ax[2].transAxes)
for c in range(4):
    for r in range(2):
        ax[2].add_patch(plt.Rectangle((0.05+c*0.11,0.32-r*0.09),0.1,0.08,fc=cols[c],alpha=0.7,transform=ax[2].transAxes))
ax[2].text(0.5,0.06,"同色=同列。列式让'只读某列'和'高压缩'成为可能",ha="center",fontsize=8,style="italic",transform=ax[2].transAxes)
plt.tight_layout(); plt.savefig("/tmp/big08_viz.png",dpi=80); plt.show()
print("列式:同一列连续存 → 只读需要的列 + 同类型数据高压缩 + 行组统计跳过无关数据")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **存储格式的选择,常比算法优化更能提速**:真实基准里,仅仅把 CSV 换成 Parquet,读全表快了 15 倍、只读一列快了 35 倍、体积小了 3 倍——**没改一行分析逻辑**。原因是列式存储天生契合分析:只读要的列(列裁剪)、同列数据高压缩、行组统计支持跳过无关数据(谓词下推)。很多人花大力气优化 pandas 代码,却忽略了**最大的免费加速就是把数据存成 Parquet**。这也解释了为什么所有数据湖、数仓、大数据系统的底层存储都是 Parquet/ORC 这类列式格式,而不是 CSV。
2. **Parquet 管磁盘,Arrow 管内存,二者是一对**:①**Parquet** 是**磁盘上**的持久化格式——压缩、带 schema、带统计,为"存得小、查得快"优化。②**Arrow** 是**内存里**的列式标准——它的意义是让 pandas、Polars、DuckDB、Spark **共用同一种内存格式**,于是数据在它们之间流转时**零拷贝、零序列化**(我们看到一份 Arrow 内存被 Polars 直接接手、被 DuckDB 直接查 SQL)。在 Arrow 出现之前,每个工具都有自己的内存格式,跨工具就得反复"序列化→反序列化",慢且费内存。Arrow 成了现代数据栈的"内存通用语",这是 Polars/DuckDB 能无缝互操作的底层原因。
3. **诚实的边界:没有万能格式,按访问模式选**。①**CSV 并非一无是处**——它人类可读、任何工具都认、适合小数据交换和给非技术同事看;只是**别用它存大数据**。②**列式不是永远最优**:如果你的访问模式是"频繁地一次读写整行、随机更新单条记录"(OLTP),行式(数据库、Avro)更合适——列式改一行要碰所有列块。③**Avro 在流式场景是对的**:Kafka 消息、需要 schema 演进(加字段不破坏旧数据)的场景,行式的 Avro 比 Parquet 更合适,因为它逐行写、schema 演进友好。④**分区(partitioning)** 是 Parquet 的重要实战技巧:按日期/地区把数据分目录存,查询时只扫相关目录(分区裁剪),能再快一个量级。**结论:分析和存储用 Parquet(配分区)、内存交换用 Arrow、流式和 schema 演进用 Avro、给人看用 CSV——按数据的访问模式而不是流行度选格式。**

**English**:
1. **Choosing the storage format often speeds things up more than algorithm optimization**: in the real benchmark, merely switching CSV to Parquet made full reads 15x faster, single-column reads 35x faster, and files 3x smaller — **without changing a line of analysis logic**. Columnar storage inherently fits analytics: read only needed columns (pruning), high same-column compression, and row-group statistics to skip irrelevant data (predicate pushdown). Many pour effort into optimizing pandas code while ignoring that **the biggest free speedup is storing data as Parquet**. This is why all data lakes, warehouses, and big-data systems use columnar Parquet/ORC underneath, not CSV.
2. **Parquet is for disk, Arrow for memory — a pair**: ① **Parquet** is the **on-disk** persistence format — compressed, with schema and statistics, optimized for "store small, query fast." ② **Arrow** is the **in-memory** columnar standard — its point is letting pandas, Polars, DuckDB, Spark **share one memory format**, so data flows among them **zero-copy, zero-serialization** (we saw one Arrow buffer adopted directly by Polars and queried directly by DuckDB with SQL). Before Arrow, each tool had its own memory format, and crossing tools meant repeated "serialize → deserialize," slow and memory-heavy. Arrow became the modern stack's "in-memory lingua franca," the underlying reason Polars/DuckDB interoperate seamlessly.
3. **Honest limits: no universal format — choose by access pattern**. ① **CSV isn't worthless** — it's human-readable, understood by any tool, good for small interchange and showing non-technical colleagues; just **don't store big data in it**. ② **Columnar isn't always optimal**: if your access pattern is "frequently read/write whole rows, randomly update single records" (OLTP), row-based (databases, Avro) fits better — updating one row in columnar touches all column chunks. ③ **Avro is right for streaming**: for Kafka messages and scenarios needing schema evolution (add fields without breaking old data), row-based Avro beats Parquet because it writes row-wise and is schema-evolution-friendly. ④ **Partitioning** is a key Parquet technique: store data in directories by date/region so queries scan only relevant directories (partition pruning), another order of magnitude faster. **Conclusion: use Parquet (with partitioning) for analytics and storage, Arrow for in-memory interchange, Avro for streaming and schema evolution, CSV for humans — choose the format by the data's access pattern, not popularity.**

> 💼 **实战视角 / Practical angle**
> **中文**:存储格式落地:①**默认把分析数据存成 Parquet**(snappy/zstd 压缩), 而非 CSV——免费提速数十倍;②**分区** `path/date=2024-01-01/region=us/` + 查询时分区裁剪, 是数据湖标配;③**Arrow 做跨工具/跨语言传输**(pandas↔Polars↔DuckDB↔Spark 零拷贝, Arrow Flight 跨网络);④**Avro 用于 Kafka/流式 + Schema Registry** 管 schema 演进;⑤ORC 在 Hive/老 Hadoop 栈里常见。**读取技巧**:只 `columns=[...]` 读需要的列、用 `filters=` 谓词下推、合理设行组大小、避免小文件过多(合并)。**面试金句**:*"大数据用列式 Parquet 不用 CSV, 因为分析只读部分列——列裁剪+高压缩+行组统计谓词下推带来数十倍加速; Parquet 是磁盘标准、Arrow 是内存标准(零拷贝跨引擎, 现代数据栈的通用语); 流式和 schema 演进用行式 Avro; 数据湖用分区 Parquet + 分区裁剪。"*
> **English**: Storage formats in practice: ① **store analytical data as Parquet by default** (snappy/zstd compression), not CSV — free tens-of-times speedup; ② **partition** `path/date=2024-01-01/region=us/` + partition pruning at query time, standard for data lakes; ③ **use Arrow for cross-tool/cross-language transfer** (pandas↔Polars↔DuckDB↔Spark zero-copy, Arrow Flight over the network); ④ **Avro for Kafka/streaming + a Schema Registry** to manage schema evolution; ⑤ ORC common in Hive/older Hadoop stacks. **Read tips**: read only needed `columns=[...]`, use `filters=` for predicate pushdown, set sensible row-group sizes, avoid too many small files (compact them). Interview line: *"Big data uses columnar Parquet not CSV because analytics reads only some columns — column pruning + high compression + row-group-statistics predicate pushdown give tens-of-times speedup; Parquet is the on-disk standard, Arrow the in-memory one (zero-copy cross-engine, the modern stack's lingua franca); use row-based Avro for streaming/schema evolution; data lakes use partitioned Parquet + partition pruning."*

---
### 小结 / Summary
- **中文**:行式(CSV/Avro)整行连续、适合事务/流式; 列式(Parquet/ORC/Arrow)整列连续、适合分析(列裁剪+高压缩+谓词下推)。
- **English**: Row-based (CSV/Avro) whole-row contiguous, good for transactions/streaming; columnar (Parquet/ORC/Arrow) whole-column contiguous, good for analytics (column pruning + high compression + predicate pushdown).
- **中文**:真实基准:CSV→Parquet 读全表快 15x、只读1列快 35x、体积小 3x(没改分析逻辑)。
- **English**: Real benchmark: CSV→Parquet is 15x faster full-read, 35x faster single-column, 3x smaller (no logic change).
- **中文**:Parquet=磁盘列式标准, Arrow=内存列式标准(零拷贝跨引擎); 流式/schema演进用 Avro; 数据湖用分区 Parquet。
- **English**: Parquet = on-disk columnar standard, Arrow = in-memory columnar standard (zero-copy cross-engine); Avro for streaming/schema evolution; data lakes use partitioned Parquet.
